**Template base con nomenclatura:**

*   Problema: detección de cáncer
*   Algoritmo: random forest

In [ ]:
import joblib
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer

# --- 1. Cargar y Entrenar ---
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

print(f"Accuracy en Test: {model.score(X_test, y_test):.4f}")

# --- 2. Definir Nomenclatura ---
fecha_hoy = datetime.now().strftime("%Y%m%d")
tipo_modelo = "RandomForest"
caso_uso = "DiagnosticoCancer"

nombre_archivo = f"{fecha_hoy}_{tipo_modelo}_{caso_uso}.joblib"

# --- 3. Guardar el Modelo ---
print(f"Guardando modelo en: {nombre_archivo}")
joblib.dump(model, nombre_archivo)

# --- 4. Cargar y Usar en Producción ---
loaded_model = joblib.load(nombre_archivo)
# loaded_model.predict(nuevos_datos)

Accuracy en Test: 0.9720
Guardando modelo en: 20260422_RandomForest_DiagnosticoCancer.joblib


# **AutoML**

Utilizando los 14 meses del dataset https://drive.google.com/drive/folders/1eizE7zkFswKzm-jog9n8HvKoh0zDRmM_?usp=drive_link

Conseguir un modelo estable (5% dentro del train_test_val), No debe haber decaimiento en OOT mayor al 10%.

CV menor a 5%

Los umbrales deben ser parámetros de sistema

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Preprocesamiento**


In [ ]:
import glob
import numpy as np
import pandas as pd
import dask.dataframe as dd
# polar, dask, pandas, wrangler son librerias de lectura.
# dask no carga los dataset en memoria sino que ejecuta la operación requerida solo sobre las columnas necesarias.

from sklearn.model_selection import train_test_split

# RUTAS DE ENTRADA DEL JOB ====================================================

DIR_RAWDATA = '/content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Dataset/Data Cruda'# Dirección de inferencia - un mes (partición de datos) a la vez
DIR_TRAINING = '/content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Datos/Data Entrenamiento' # Dirección donde tengamos acumulado meses de entrenamiento

# RUTAS DE SALIDA DEL JOB =====================================================

DIR_PROCESSED = '/content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Datos/Data Preprocesada' # Salida es la data procesada

# LEER VARIABLES CRUDAS =======================================================

def read_rawdata(period, type_work):
    if type_work == 'training':
        csv_files = glob.glob(f'{DIR_TRAINING}/*.csv') #.parquet - método de partición de datos
        df_list = []
        for csv_file in csv_files:
            try:
                df = pd.read_csv(csv_file)
                df_list.append(df)
            except Exception as e:
                print(f"Error reading {csv_file}: {e}")

        if df_list:
            df = pd.concat(df_list, ignore_index=True)
            print("Successfully unified dataframes.")
        else:
            df = pd.DataFrame()
            print("No CSV files found or read errors occurred.")

    elif type_work == 'inference':
        try: # Intentar leer las variables en formato 'csv'
            path = glob.glob(f'{DIR_RAWDATA}/p{period}_extrac.csv')[0]
            df = pd.read_csv(path)
        except: # Intentar leer las variables en formato 'parquet'
            path = f'{DIR_RAWDATA}/*'
            df = dd.read_parquet(path)
            df = df.compute().reset_index(drop=True)
    return df

# PROCESAR VARIABLES ==========================================================

def one_hot_encoding(df, col, categories=None):
    selected_col = df[col]

    if categories is not None: # Forzar las categorias del one hot encoding
        selected_col = selected_col.astype(pd.CategoricalDtype(categories))

    new_cols = pd.get_dummies(selected_col, prefix=col)
    df = df.drop([col], axis='columns')
    df = pd.concat([df, new_cols], axis='columns')

    return df, new_cols

def process_vars(df):
    variables_categoricas = ['ent_1erlntcrallsfm01']
    variables_numericas = ['nro_producto_6m', 'prom_uso_tc_rccsf3m', 'ctd_sms_received',
                           'max_usotcribksf06m', 'ctd_camptot06m', 'dsv_svppallsf06m',
                           'prm_svprmecs06m', 'ctd_app_productos_m1', 'ctd_campecsm01',
                           'lin_tcrrstsf03m', 'mnt_ptm', 'dif_no_gestionado_4meses',
                           'max_campecs06m', 'beta_pctusotcr12m', 'rat_disefepnm01',
                           'flg_saltotppe12m', 'prom_sow_lintcribksf3m', 'openhtml_1m',
                           'nprod_1m', 'nro_transfer_6m', 'max_usotcrrstsf03m',
                           'prm_cnt_fee_amt_u7d', 'pas_avg6m_max12m', 'beta_saltotppe12m',
                           'seg_un', 'ant_ultprdallsf', 'avg_sald_pas_3m', 'pas_1m_avg3m',
                           'num_incrsaldispefe06m', 'cnl_age_p4m_p12m', 'cnl_atm_p4m_p12m',
                           'cre_lin_tc_rccibk_m07', 'prm_svprmlibdis06m', 'ingreso_neto',
                           'max_nact_12m', 'cre_sldtotfinprm03', 'dif_contacto_efectivo_10meses',
                           'act_1m_avg3m', 'monto_consumos_ecommerce_tc', 'ctd_camptotm01',
                           'prop_atm_4m', 'prom_pct_saldopprcc6m', 'apppag_1m', 'nro_configuracion_6m',
                           'act_avg6m_max12m', 'sldvig_tcrsrcf', 'prom_score_acepta_12meses',
                           'telefonos_6meses', 'pas_1m_avg6m', 'ctd_camptototrcnl06m',
                           'prm_saltotrdpj03m', 'bpitrx_1m', 'prm_lintcribksf03m', 'ctd_entrdm01',
                           'avg_openhtml_6m', 'tea', 'pct_usotcrm01','senthtml_1m']

    df = df.replace(['', 'null', 'None'], [np.nan, np.nan, np.nan])

    for column in variables_numericas:
        df[column] = df[column].fillna(-9999999)

    df = df.astype({v: 'float32' for v in variables_numericas})
    df = df.astype({v: 'string' for v in variables_categoricas})
    df = df.astype({v: 'string' for v in ['partition']})

    Variables_faltanLlenarNulos = {'ent_1erlntcrallsfm01': ['INTERBANK']}

    for a in Variables_faltanLlenarNulos:
        df[a] = df[a].fillna('SV')
        df.loc[~df[a].isin(Variables_faltanLlenarNulos[a]), a] = 'OTRO'

    cols_dumm = pd.DataFrame()
    for col in Variables_faltanLlenarNulos:
        default_value = 'OTRO'
        values = Variables_faltanLlenarNulos[col]
        values.append(default_value)
        df, dummy = one_hot_encoding(df, col, values)
        cols_dumm = pd.concat([cols_dumm, dummy], axis='columns')

    return df

# GUARDAR SALIDAS DEL JOB =====================================================

def save_outputs(DIR_PROCESSED, df, period, model, type_work):
    col_target = 'target'
    cols_post = ['partition', 'key_value', 'codunicocli', 'grp_campecs06m', 'prob_value_contact', 'monto']
    cols_vars = ['nro_producto_6m', 'prom_uso_tc_rccsf3m', 'ctd_sms_received',
                 'max_usotcribksf06m', 'ctd_camptot06m', 'dsv_svppallsf06m',
                 'prm_svprmecs06m', 'ctd_app_productos_m1', 'ctd_campecsm01',
                 'lin_tcrrstsf03m', 'mnt_ptm', 'dif_no_gestionado_4meses',
                 'max_campecs06m', 'beta_pctusotcr12m', 'rat_disefepnm01',
                 'flg_saltotppe12m', 'prom_sow_lintcribksf3m', 'openhtml_1m', 'nprod_1m',
                 'nro_transfer_6m', 'max_usotcrrstsf03m', 'prm_cnt_fee_amt_u7d',
                 'pas_avg6m_max12m', 'beta_saltotppe12m', 'seg_un', 'ant_ultprdallsf',
                 'avg_sald_pas_3m', 'pas_1m_avg3m', 'num_incrsaldispefe06m',
                 'cnl_age_p4m_p12m', 'cnl_atm_p4m_p12m', 'cre_lin_tc_rccibk_m07',
                 'prm_svprmlibdis06m', 'ingreso_neto', 'max_nact_12m',
                 'cre_sldtotfinprm03', 'dif_contacto_efectivo_10meses', 'act_1m_avg3m',
                 'monto_consumos_ecommerce_tc', 'ctd_camptotm01', 'prop_atm_4m',
                 'prom_pct_saldopprcc6m', 'apppag_1m', 'nro_configuracion_6m',
                 'act_avg6m_max12m', 'sldvig_tcrsrcf', 'prom_score_acepta_12meses',
                 'telefonos_6meses', 'pas_1m_avg6m', 'ctd_camptototrcnl06m',
                 'prm_saltotrdpj03m', 'bpitrx_1m', 'prm_lintcribksf03m', 'ctd_entrdm01',
                 'avg_openhtml_6m', 'tea', 'pct_usotcrm01', 'senthtml_1m',
                 'ent_1erlntcrallsfm01_INTERBANK', 'ent_1erlntcrallsfm01_OTRO']

    datasets = [('', df, None)] # type_work == 'inference'
    period = str(period) + '_'

    if type_work == 'training':
        cols = list(dict.fromkeys(cols_vars + cols_post))

        x_train, x_test, \
        y_train, y_test = train_test_split(df[cols],
                                           df[col_target],
                                           test_size=0.33,
                                           random_state=123)

        datasets = [('train_', x_train, y_train), ('test_', x_test, y_test)]
        period = ''
        DIR_PROCESSED = DIR_PROCESSED + '/training_data'

    for prefix, x, y in datasets:
        path = f'{DIR_PROCESSED}/preprocessed/{prefix}vars_{period}{model}.csv'
        pd.concat([y, x[cols_vars]], axis=1).to_csv(path, index=False)

        path = f'{DIR_PROCESSED}/postprocessed/{prefix}post_{period}{model}.csv'
        x[cols_post].to_csv(path, index=False)

# CODIGO PRINCIPAL ============================================================

def main(DIR_PROCESSED, type_work, period = ''):
    df = read_rawdata(period, type_work)
    df = process_vars(df)
    save_outputs(DIR_PROCESSED, df, period, 'extrac', type_work)


In [ ]:
if __name__ == '__main__':
    main(DIR_PROCESSED, 'training')

Successfully unified dataframes.


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.6 MB/s eta 0:00:00


**Entrenamiento**

In [ ]:
from datetime import datetime
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import catboost as catb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pickle
import time
import json
import sys
import platform

In [ ]:
# Define the paths to the datasets
train_path = '/content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Datos/Data Preprocesada/training_data/preprocessed/train_vars_extrac.csv'
test_path = '/content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Datos/Data Preprocesada/training_data/preprocessed/test_vars_extrac.csv'
model_save_dir = '/content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Datos/Modelo'

In [ ]:
def preprocess_dataframe(df):
    for col in df.columns:
        if df[col].dtype == 'bool':
            df[col] = df[col].astype(int)
        elif df[col].dtype in ['int16', 'int32', 'int64']:
            df[col] = df[col].astype(int)
        elif df[col].dtype in ['float16', 'float32', 'float64']:
            # floats con solo 4 decimales
            df[col] = df[col].astype(float).round(4)
    return df

def auto_train_v1(train_path, test_path, model_save_dir):
    now = datetime.now()
    folder_name = now.strftime("%Y-%m-%d_%H-%M-%S") #Donde se guardan los modelos
    model_save_dir = f'{model_save_dir}/{folder_name}'
    os.makedirs(model_save_dir, exist_ok=True)
    print(f"Directorio de modelos guardado en: {model_save_dir}")
    # cargar data
    try:
        train_df = pd.read_csv(train_path)
        test_df = pd.read_csv(test_path)
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo del conjunto de datos. Revisar las rutas: {train_path}, {test_path}")
        return

    # Preprocess data
    train_df = preprocess_dataframe(train_df)
    test_df = preprocess_dataframe(test_df)

    # Separar features y target
    X_train = train_df.drop(columns=[train_df.columns[0]])
    y_train = train_df[train_df.columns[0]]
    X_test = test_df.drop(columns=[test_df.columns[0]])
    y_test = test_df[test_df.columns[0]]

    # Asegurarse que las columnas de train y test coincidan
    train_cols = X_train.columns #todo menos el target
    test_cols = X_test.columns #target

    missing_in_test = set(train_cols) - set(test_cols)
    for c in missing_in_test:
        X_test[c] = 0

    missing_in_train = set(test_cols) - set(train_cols)
    for c in missing_in_train:
        X_train[c] = 0

    X_test = X_test[train_cols]

    # Model training and evaluation
    models = {
        'xgb': {
            'model': xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
            'params': {}, # parametros por default
            'performance': {}
        },
        'lgbm': {
            'model': lgb.LGBMClassifier(random_state=42),
            'params': {}, # parametros por default
            'performance': {}
        },
        'catb': {
            'model': catb.CatBoostClassifier(verbose=0, random_state=42),
            'params': {}, # parametros por default
            'performance': {}
        }
    }

    best_ml_name = None
    best_auc_test = -np.inf
    best_decay = np.inf

    for ml_name, model_info in models.items():
        model = model_info['model']
        start_time = time.time()
        if ml_name == 'catb':
             # CatBoost puede manejar diferentes tipos de datos, pero usemos los datos preprocesados
             model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=10, verbose=0)
        else:
            model.fit(X_train, y_train)
        end_time = time.time()
        training_time = end_time - start_time

        y_train_pred_proba = model.predict_proba(X_train)[:, 1]
        y_test_pred_proba = model.predict_proba(X_test)[:, 1]

        auc_train = roc_auc_score(y_train, y_train_pred_proba)
        auc_test = roc_auc_score(y_test, y_test_pred_proba)

        decay = ((auc_train - auc_test) / auc_train) * 100 if auc_train > 0 else np.inf
        # No es el coeficiente de variación

        # Registro de modelos
        models[ml_name]['performance'] = {
            'auc_train': auc_train,
            'auc_test': auc_test,
            'decay_percent': decay,
            'training_time_segs': training_time
        }
        models[ml_name]['params'] = model.get_params() # Para la reproductibilidad del entrenamiento

        print(f"Model: {ml_name}")
        print(f"  AUC Train: {auc_train:.4f}")
        print(f"  AUC Test: {auc_test:.4f}")
        print(f"  Decay (%): {decay:.2f}%")
        print("-" * 20)

        # Check if this model is the champion | prioriza la estabilidad | métricas de negocio
        if auc_test > best_auc_test and decay < 10:
            best_auc_test = auc_test
            best_decay = decay
            best_ml_name = ml_name

    if best_ml_name:
        print(f"\nModelo finalista: {best_ml_name}")
        save_model(models[best_ml_name]['model'], best_ml_name, models[best_ml_name]['performance'], models[best_ml_name]['params'], model_save_dir)
    else:
        print("\nNo se encontró ningún modelo campeón que cumpla con los criterios (Decaimiento < 10%).")

def save_model(model, ml_name, performance, params, save_dir):
    # Save the model
    model_filename = f'{ml_name}_model.pkl'
    model_path = f'{save_dir}/{model_filename}'
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f"Modelo guardado en: {model_path}")

    # Save metadata
    metadata_filename = f'{ml_name}_metadata.json'
    metadata_path = f'{save_dir}/{metadata_filename}'

    # Get library versions
    # Importante para la reproductibilidad (auditorias)
    library_versions = {
        'xgboost': xgb.__version__ if ml_name == 'xgb' else None,
        'lightgbm': lgb.__version__ if ml_name == 'lgbm' else None,
        'catboost': catb.__version__ if ml_name == 'catb' else None,
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'scikit-learn': platform.version() # Esto podría ser útil para obtener información general del sistema.
    }

    metadata = {
        'ml_name': ml_name,
        'performance': performance,
        'hyperparameters': params,
        'library_versions': library_versions,
        'timestamp': time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
    }

    # Limpiar parámetros para la serialización JSON (por ejemplo, manejar tipos no serializables)
    def clean_params(p):
        cleaned = {}
        for k, v in p.items():
            try:
                json.dumps(v) # Check if serializable
                cleaned[k] = v
            except (TypeError, OverflowError):
                cleaned[k] = str(v)
        return cleaned

    metadata['hyperparameters'] = clean_params(metadata['hyperparameters'])

    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=4)
    print(f"Metadata guardada en: {metadata_path}")


# Run the training and comparison
auto_train_v1(train_path, test_path, model_save_dir)

Directorio de modelos guardado en: /content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Datos/Modelo/2026-04-22_02-25-25


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:25:57] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Model: xgb
  AUC Train: 0.8882
  AUC Test: 0.8476
  Decay (%): 4.57%
--------------------
[LightGBM] [Info] Number of positive: 33815, number of negative: 1025262
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.681940 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9397
[LightGBM] [Info] Number of data points in the train set: 1059077, number of used features: 60
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.031929 -> initscore=-3.411799
[LightGBM] [Info] Start training from score -3.411799
Model: lgbm
  AUC Train: 0.8683
  AUC Test: 0.8529
  Decay (%): 1.77%
--------------------
Model: catb
  AUC Train: 0.8587
  AUC Test: 0.8525
  Decay (%): 0.73%
--------------------

Modelo finalista: lgbm
Modelo guardado en: /content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Datos/Modelo/2026-04-22_02-25-25/lgbm_mod